# Using a useful subpackage of regional_mom6 - mom6_forge!

For this example we need:

- [GEBCO bathymetry](https://www.gebco.net/data_and_products/gridded_bathymetry_data/)

## What does the `mom6_forge` package do?

mom6_forge underwrites the grid-generation and bathymetry-generation parts of regional_mom6. Previously, regional_mom6 had all of this inside of itself. It's now part of mom6_forge to take advantage of a few cool features inside of that package, some of which we'll show below! If you're interested in more features than shown here (things like slicing grids, filling bathymetry with a spoon-type depth, writing CICE grids), look at mom6_forge docs! We're up to date with them!

## What does this notebook do?
This notebook shows two ways of getting a `regional_mom6.experiment` its horizontal grid, vertical grid, and bathymetry:

1. **Let the experiment build them for you** (the default, shown in the other demos): pass extents/resolution/depth and call `setup_bathymetry`.
2. **Build the mom6_forge objects yourself first**, then hand them to the experiment. This is useful if you want to use the interactive `GridCreator`/`VGridCreator`/`TopoEditor` widgets, or if you already have a grid/vgrid/topo from another case.

Input Type | Source | Subsets required
---|---|---
Bathymetry | [GEBCO](https://www.gebco.net/data_and_products/gridded_bathymetry_data/) | whole globe or subset around domain

In [ ]:
%load_ext autoreload
%autoreload 2
import warnings
warnings.filterwarnings("ignore")
import regional_mom6 as rmom6
from mom6_forge.grid import Grid
from mom6_forge.vgrid import VGrid
from mom6_forge.topo import Topo
from pathlib import Path
import os

## Step 1: Choose our domain, define workspace paths

To make sure that things are working I'd recommend starting with the default example defined below. If this runs ok, then change to a domain of your choice and hopefully it runs ok too! If not, check the [README](https://github.com/COSIMA/regional-mom6/blob/main/README.md) and [documentation](https://regional-mom6.readthedocs.io/) for troubleshooting tips.

You can log in and use [this GUI](https://data.marine.copernicus.eu/product/GLOBAL_MULTIYEAR_PHY_001_030/download) to find the lat/lon of your domain and copy paste below.

In [ ]:
expt_name = "tasmania-example-reanalysis"

latitude_extent = [-48, -38.95]
longitude_extent = [143, 150]

date_range = ["2003-01-01 00:00:00", "2003-01-05 00:00:00"]

resolution = 0.05
number_vertical_layers = 75
layer_thickness_ratio = 60
depth = 4500
minimum_depth = 5

## Place where all your input files go
input_dir = Path(f"mom6_input_directories/{expt_name}/")

## Directory where you'll run the experiment from
run_dir = Path(f"mom6_run_directories/{expt_name}/")

## if directories don't exist, create them
for path in (run_dir, input_dir):
    os.makedirs(str(path), exist_ok=True)

## Step 2: Build the mom6_forge grid objects yourself

A `regional_mom6.experiment` is really just a thin wrapper around three mom6_forge objects: a `Grid` (horizontal grid), a `VGrid` (vertical grid), and a `Topo` (bathymetry). Once the experiment has one of these, it's stored as `expt.m6f_hgrid`, `expt.m6f_vgrid`, and `expt.m6f_bathymetry` respectively, and it's the single source of truth -- `expt.hgrid`, `expt.vgrid`, and `expt.bathymetry` are just `xarray.Dataset` views generated from them on demand (more on that below).

You don't have to build these yourself -- pass extents/resolution and the experiment builds them for you, like in the other demos. But you can also build them ahead of time and hand them in directly. This is handy if you want to use the interactive creator widgets below, or if you're reusing a grid from elsewhere.

Two ways to build a `Grid`:
1. **In code**, with the same constructor the experiment uses internally.
2. **Interactively**, with the `GridCreator` widget -- drag a rectangle on a map, or set a center/width/height, or use a projected CRS.

In [ ]:
# 1. In code -- this mirrors exactly what `experiment(hgrid_type="even_spacing", ...)` does internally
hgrid = Grid(
    resolution=resolution,
    xstart=longitude_extent[0],
    lenx=longitude_extent[1] - longitude_extent[0],
    ystart=latitude_extent[0],
    leny=latitude_extent[1] - latitude_extent[0],
    name=expt_name,
    type="rectilinear_cartesian",
)
hgrid

In [ ]:
# 2. Interactively, with the GridCreator widget.
# Select a creation method (Lat/Lon Corners, From Center, or From Projection), then
# press "Select Region" and interact with the map. Grids are saved under
# <working_dir>/GridLibrary/ as you go.
%matplotlib ipympl
from mom6_forge.grid_creator import GridCreator

GridCreator(working_dir=input_dir)

# This does not render in the documentation, but on a local jupyter notebook it will
# show an interactive map. If you're having issues on JupyterHub, make sure the
# ipympl extension is installed.

Same idea for the vertical grid -- build a `VGrid` in code, or interactively with `VGridCreator`.

In [ ]:
# 1. In code -- mirrors `experiment(vgrid_type="hyperbolic_tangent", ...)`
vgrid = VGrid.hyperbolic(
    nk=number_vertical_layers,
    depth=depth,
    ratio=layer_thickness_ratio,
    name=expt_name,
)
vgrid.dz

In [ ]:
# 2. Interactively, with the VGridCreator widget. Passing `topo=` lets it infer a
# sensible min_depth if you've already got a Topo object; here we don't yet, so it
# just uses the vgrid we already built as a starting point.
%matplotlib ipympl
from mom6_forge.vgrid_creator import VGridCreator

VGridCreator(vgrid=vgrid)

## Step 3: Make the experiment object, passing in the grids we just built

Pass `hgrid_type`/`vgrid_type` a `Grid`/`VGrid` object directly (instead of `"even_spacing"`/`"hyperbolic_tangent"`) and the experiment stores it as `expt.m6f_hgrid`/`expt.m6f_vgrid` without touching disk. `longitude_extent`/`latitude_extent`/`resolution`/`number_vertical_layers`/`layer_thickness_ratio` are inferred from the objects, so they don't need to be passed again.

In [ ]:
expt = rmom6.experiment(
    date_range=date_range,
    resolution=None,
    number_vertical_layers=None,
    layer_thickness_ratio=None,
    depth=depth,
    minimum_depth=minimum_depth,
    mom_run_dir=run_dir,
    mom_input_dir=input_dir,
    hgrid_type=hgrid,
    vgrid_type=vgrid,
)

### How the `hgrid`/`vgrid`/`bathymetry` properties work

`expt.hgrid`, `expt.vgrid`, and `expt.bathymetry` are properties, not plain attributes. Each one is **always regenerated live** from its mom6_forge object (`expt.m6f_hgrid`, `expt.m6f_vgrid`, `expt.m6f_bathymetry`) -- there's no separate cached copy that can go stale. Concretely:

- `expt.hgrid` returns `expt.m6f_hgrid.supergrid.to_ds()`
- `expt.vgrid` returns `expt.m6f_vgrid.write_z_file(mom_input_dir/"vcoord.nc")` (this also rewrites `vcoord.nc` each time it's accessed)
- `expt.bathymetry` returns `expt.m6f_bathymetry.gen_topo_ds()`

That means if you edit `expt.m6f_hgrid`/`expt.m6f_vgrid`/`expt.m6f_bathymetry` in place -- e.g. via `TopoEditor` below -- the very next read of `expt.hgrid`/`expt.vgrid`/`expt.bathymetry` reflects the edit immediately, with no extra step needed to "refresh" it.

If a `m6f_*` object hasn't been supplied yet (you didn't pass one in, and haven't called `setup_bathymetry` yet), the property lazily builds one by reading the corresponding file from `mom_input_dir` (`hgrid.nc`, `vgrid.nc`, or `bathymetry.nc`) the first time it's accessed -- this is the same thing the old `hgrid_type="from_file"` option did explicitly, it now just happens automatically whenever the object hasn't been built or passed in yet.

## Step 4: Set up bathymetry

Similarly to ocean forcing, we point the experiment's `setup_bathymetry` method at the location of the file of choice and also provide the variable names. We don't need to preprocess the bathymetry since it is simply a two-dimensional field and is easier to deal with. Afterwards you can inspect `expt.bathymetry` to have a look at the regional domain.

Under the hood, this builds a `Topo` object from `bathymetry_path` and stores it as `expt.m6f_bathymetry`.

After running this cell, your input directory will contain other bathymetry-related things like the ocean mosaic and mask table too. The mask table defaults to a 10x10 layout and can be modified later.

In [ ]:
expt.setup_bathymetry(
    bathymetry_path='PATH_TO_GEBCO_FILE/GEBCO_2022.nc',
    longitude_coordinate_name='lon',
    latitude_coordinate_name='lat',
    vertical_coordinate_name='elevation',
    )

### Try out the topo editor!

`expt.m6f_bathymetry` is the mom6_forge `Topo` object, which can be used in the topo editor.

In [ ]:
%matplotlib ipympl
from mom6_forge.topo_editor import TopoEditor

TopoEditor(expt.m6f_bathymetry)

# This does not render in the documentation, but on a local jupyter notebook it will show an interactive map where you can edit the bathymetry. If you are having issues on JupyterHub, please make sure the ipympl extension is installed in the JupyterHub

In [ ]:
# `expt.bathymetry` already reflects your edits (it's regenerated live from
# `expt.m6f_bathymetry`), but you still need to write it to disk for MOM6 to use it:
expt.m6f_bathymetry.write_topo(expt.mom_input_dir / "bathymetry.nc")

### You can also build (or edit) a `Topo` object yourself, before calling `setup_bathymetry`

There's no `bathymetry_type` constructor argument -- instead, just assign a `Topo` object straight to `expt.m6f_bathymetry`, the same way `hgrid_type`/`vgrid_type` work for the grids:

In [ ]:
# Builds an empty Topo (all NaN depth) tied to expt's current horizontal grid, which
# you could then fill in via TopoEditor, `set_depth_from_stats`, `set_from_dataset`, etc.
expt.m6f_bathymetry = Topo(grid=expt.m6f_hgrid, min_depth=minimum_depth, git=False)

## Writing out alternate files!

ESMF file: If you run with the NUOPC Coupler, you need to generate an ESMF File, which contains all kinds of grid and mask information.

CICE file: If you run with the sea-ice model CICE, you need to generate a CICE specific file written on a Arakawa B grid. (until they update to use MOM6 style grids/bathy!)

In [ ]:
# Write out an ESMF File
expt.m6f_bathymetry.write_esmf_mesh(expt.mom_input_dir / "ESMF_grid.nc")

# Write out a CICE grid File
expt.m6f_bathymetry.write_cice_grid(expt.mom_input_dir / "CICE_grid.nc")

## Recap: object names

| mom6_forge engine object (single source of truth) | `xarray.Dataset` view (regenerated live, for plotting/inspection) |
|---|---|
| `expt.m6f_hgrid` (a `Grid`) | `expt.hgrid` |
| `expt.m6f_vgrid` (a `VGrid`) | `expt.vgrid` |
| `expt.m6f_bathymetry` (a `Topo`) | `expt.bathymetry` |

Always edit the `m6f_*` object (directly, or via `GridCreator`/`VGridCreator`/`TopoEditor`), then read back the corresponding plain-named property to see an `xarray.Dataset` of the result. Keep in mind `expt.m6f_bathymetry` is tied to whichever `expt.m6f_hgrid` it was built from -- if you change `expt.m6f_hgrid` afterwards, you need to rebuild `expt.m6f_bathymetry` (e.g. by calling `setup_bathymetry` again, or re-running the cell above).

In [ ]:
# Resaving files directly from the mom6_forge objects
expt.m6f_hgrid.write_supergrid(expt.mom_input_dir / "hgrid.nc")
expt.m6f_bathymetry.write_topo(expt.mom_input_dir / "bathymetry.nc")
expt.m6f_vgrid.write_z_file(expt.mom_input_dir / "vcoord.nc")

## Other Small Features We Can Use Now
 - Access grid t/u/v/q points easily with: `expt.m6f_hgrid.[t/u/v/q]lon`
 - Access bathymetry mask at t/u/v/q points easily with: `expt.m6f_bathymetry.[t/u/v/q/supergrid]mask`
 - You can write a vgrid as a thickness file instead of at interfaces with `expt.m6f_vgrid.write(expt.mom_input_dir/"vgrid.nc")`
 - For more on `GridCreator`, `VGridCreator`, and `TopoEditor` (slicing grids, filling bathymetry, undo/redo, version control, etc.), see the [mom6_forge documentation](https://ncar.github.io/mom6_forge/).